# 📘 Deep Learning Text Generation — Week 5 Assignment (SOLVED)
## Text Generation using **Vanilla RNN, LSTM, and GRU**

This notebook is built for **students and beginners** to understand how sequence models learn:
- grammar
- sentence flow
- contextual dependencies
- next-word prediction
- text generation

🎯 **Goal:** Compare **Simple RNN vs LSTM vs GRU** on the same text corpus and understand why gated architectures perform better.

# 🧠 Problem Statement
Design and implement a DL model capable of learning the underlying structure, grammar, and contextual dependencies of a given text corpus to generate coherent and meaningful text sequences using:

1. **Vanilla RNN**
2. **LSTM**
3. **GRU**

Then compare:
- training loss
- generated text quality
- memory handling
- long-term dependency learning

In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, GRU, Dense
import numpy as np
import matplotlib.pyplot as plt
print("TensorFlow:", tf.__version__)

TensorFlow: 2.21.0


# 📥 Load Text Corpus
We use a **small built-in sample corpus** so students can run this quickly.
You can later replace it with Shakespeare text, song lyrics, chatbot data, story paragraphs, or custom PDF extracted text.

In [2]:
corpus = '''
deep learning is transforming artificial intelligence
recurrent neural networks are useful for sequential data
lstm helps remember long term dependencies
gru is faster and simpler than lstm
text generation models predict the next word
deep learning models can generate meaningful sentences
'''
print(corpus)


deep learning is transforming artificial intelligence
recurrent neural networks are useful for sequential data
lstm helps remember long term dependencies
gru is faster and simpler than lstm
text generation models predict the next word
deep learning models can generate meaningful sentences



# 🔤 Tokenization & Sequence Creation
We convert text into integer tokens and create **n-gram style sequences** for next-word prediction.

**How it works:**
- Each word gets a unique integer ID
- For every line, we create progressively longer sequences: `[w1,w2]`, `[w1,w2,w3]`, ...
- `X` = all words except the last → **input**
- `y` = the last word → **label to predict**

In [3]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([corpus])

total_words = len(tokenizer.word_index) + 1
print("Vocabulary size:", total_words)

input_sequences = []
for line in corpus.split('\n'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_seq = token_list[:i+1]
        input_sequences.append(n_gram_seq)

max_len = max(len(seq) for seq in input_sequences)
input_sequences = pad_sequences(input_sequences, maxlen=max_len, padding='pre')

X = input_sequences[:, :-1]
y = input_sequences[:, -1]

print("X shape:", X.shape)
print("y shape:", y.shape)

Vocabulary size: 37
X shape: (35, 7)
y shape: (35,)


# 🧠 Model 1: Vanilla RNN

**Architecture:** `Embedding → SimpleRNN(64) → Dense(softmax)`

**How it works:**
$$h_t = \tanh(W_h h_{t-1} + W_x x_t + b)$$

The hidden state $h_t$ is simply a weighted combination of the previous hidden state and current input.

**Limitation:** Suffers from **vanishing gradients** — gradients shrink exponentially as they flow backward through time, making it hard to learn long-term dependencies.

In [4]:
rnn_model = Sequential([
    Embedding(total_words, 32, input_length=max_len-1),
    SimpleRNN(64),
    Dense(total_words, activation='softmax')
])

rnn_model.compile(loss='sparse_categorical_crossentropy',
                  optimizer='adam',
                  metrics=['accuracy'])

rnn_history = rnn_model.fit(X, y, epochs=100, verbose=0)
print("Vanilla RNN training completed")
print(f"Final Loss: {rnn_history.history['loss'][-1]:.4f}  |  Final Accuracy: {rnn_history.history['accuracy'][-1]*100:.2f}%")

Vanilla RNN training completed
Final Loss: 0.8338  |  Final Accuracy: 85.71%


# 🔒 Model 2: LSTM

**Architecture:** `Embedding → LSTM(64) → Dense(softmax)`

LSTM introduces a **cell state** $C_t$ (long-term memory) alongside the hidden state, controlled by 3 gates:

| Gate | Formula | Purpose |
|------|---------|--------|
| Forget $f_t$ | $\sigma(W_f [h_{t-1}, x_t] + b_f)$ | What to erase from cell state |
| Input $i_t$ | $\sigma(W_i [h_{t-1}, x_t] + b_i)$ | What new info to write |
| Output $o_t$ | $\sigma(W_o [h_{t-1}, x_t] + b_o)$ | What to expose as hidden state |

This makes LSTM **resistant to vanishing gradients** and able to capture long-range dependencies.

In [5]:
lstm_model = Sequential([
    Embedding(total_words, 32, input_length=max_len-1),
    LSTM(64),
    Dense(total_words, activation='softmax')
])

lstm_model.compile(loss='sparse_categorical_crossentropy',
                   optimizer='adam',
                   metrics=['accuracy'])

lstm_history = lstm_model.fit(X, y, epochs=100, verbose=0)
print("LSTM training completed")
print(f"Final Loss: {lstm_history.history['loss'][-1]:.4f}  |  Final Accuracy: {lstm_history.history['accuracy'][-1]*100:.2f}%")

LSTM training completed
Final Loss: 1.5397  |  Final Accuracy: 60.00%


# ⚡ Model 3: GRU

**Architecture:** `Embedding → GRU(64) → Dense(softmax)`

GRU simplifies LSTM by merging the cell state and hidden state, using only **2 gates**:

| Gate | Formula | Purpose |
|------|---------|--------|
| Reset $r_t$ | $\sigma(W_r [h_{t-1}, x_t])$ | How much past to forget |
| Update $z_t$ | $\sigma(W_z [h_{t-1}, x_t])$ | Balance old vs new info |

**Advantages over LSTM:**
- Fewer parameters → faster training
- Often comparable performance on small datasets
- No separate cell state — simpler to implement

In [6]:
gru_model = Sequential([
    Embedding(total_words, 32, input_length=max_len-1),
    GRU(64),
    Dense(total_words, activation='softmax')
])

gru_model.compile(loss='sparse_categorical_crossentropy',
                  optimizer='adam',
                  metrics=['accuracy'])

gru_history = gru_model.fit(X, y, epochs=100, verbose=0)
print("GRU training completed")
print(f"Final Loss: {gru_history.history['loss'][-1]:.4f}  |  Final Accuracy: {gru_history.history['accuracy'][-1]*100:.2f}%")

GRU training completed
Final Loss: 1.2995  |  Final Accuracy: 80.00%


## 📉 Compare Training Loss

In [7]:
plt.figure(figsize=(10,4))
plt.plot(rnn_history.history['loss'], label='RNN')
plt.plot(lstm_history.history['loss'], label='LSTM')
plt.plot(gru_history.history['loss'], label='GRU')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss Comparison")
plt.legend()
plt.show()

# ✍️ Text Generation Function

This function:
1. Tokenizes the seed text
2. Pads it to the required length
3. Gets the model's probability distribution over all words
4. Picks the **argmax** (most likely) next word
5. Appends it and repeats `next_words` times

> 💡 **Note:** Using `argmax` is called **greedy decoding**. For more creative outputs, you can use **temperature sampling** or **top-k sampling**.

In [8]:
def generate_text(model, seed_text, next_words=5):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_len-1, padding='pre')
        predicted = np.argmax(model.predict(token_list, verbose=0), axis=-1)[0]

        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted:
                output_word = word
                break
        seed_text += " " + output_word
    return seed_text

## 🧪 Generate Text Samples

In [9]:
print("RNN :", generate_text(rnn_model, "deep learning", 5))
print("LSTM:", generate_text(lstm_model, "deep learning", 5))
print("GRU :", generate_text(gru_model, "deep learning", 5))

RNN : deep learning is transforming artificial intelligence sentences
LSTM: deep learning is transforming artificial artificial intelligence
GRU : deep learning models can generate meaningful meaningful


## 📊 Final Comparison Table

| Model | Parameters | Final Loss | Final Accuracy | Gates | Long-term Memory |
|-------|-----------|-----------|---------------|-------|------------------|
| Vanilla RNN | Fewest | 0.8338 | 85.71% | None | ❌ Weak (vanishing gradient) |
| LSTM | Most | 1.5397 | 60.00% | 3 (forget, input, output) | ✅ Strong |
| GRU | Medium | 1.2995 | 80.00% | 2 (reset, update) | ✅ Good |

> ⚠️ **Why RNN has lower loss here?** The corpus is very small (6 sentences, 37 words). RNN can memorize short patterns easily. On larger datasets, LSTM and GRU will significantly outperform RNN by capturing long-range grammar.

# 📚 Student Learning Tasks — Solutions

### ✅ Beginner Tasks

**Task 1: Replace corpus with your own paragraph**
```python
corpus = '''
the sun rises in the east and sets in the west
birds fly south during winter and return in spring
mountains are formed by tectonic plate movements
'''
```

**Task 2: Increase embedding dimension**
```python
Embedding(total_words, 64, input_length=max_len-1)  # changed 32 → 64
```

**Task 3: Increase epochs to 200**
```python
rnn_model.fit(X, y, epochs=200, verbose=0)
```

**Task 4: Change hidden units 64 → 128**
```python
SimpleRNN(128)  # or LSTM(128) or GRU(128)
```

**Task 5: Generate 10 words instead of 5**
```python
generate_text(rnn_model, "deep learning", next_words=10)
```

# ✅ Conclusion

| Model | Key Insight |
|-------|-------------|
| **Vanilla RNN** | Simple, fast, good on short sequences. Fails on long dependencies due to vanishing gradients. |
| **LSTM** | Best for long-range dependencies (paragraphs, stories). More parameters, slower to train. |
| **GRU** | Best balance — nearly as good as LSTM at capturing dependencies, but faster to train with fewer parameters. |

### 🔑 Key Takeaways
- **Vanishing gradient problem** is why plain RNN fails on longer texts
- **Gating mechanisms** (LSTM/GRU) solve this by creating gradient highways
- On small corpora, RNN may overfit and appear to win — always test on larger data
- **GRU is the practical choice** for most sequence tasks given its speed-accuracy tradeoff
- This notebook demonstrates **character/word-level language modelling** — the foundation of modern LLMs